# EEG Dementia Classification — Google Colab Pipeline

Runs feature extraction and/or model training for this repo in the cloud (no local PC needed).

**Before you start**
1. Runtime → Change runtime type → **CPU** is enough (GPU not required).
2. Pick a mode in the config cell below:
   - `ingest_only` — extract features from raw EEG, save to Drive (resumable; no training)
   - `full` — ingest + train XGBoost + MLP
   - `train_only` — load saved features from Drive or Zenodo, then train
   - `smoke_test` — process 2 subjects per dataset (quick sanity check)
3. For ingest modes, upload raw EEG to Google Drive (see data cell).

Dataset: [Miltiadous et al. 2023](https://doi.org/10.3390/data8060095)

In [ ]:
# @title Configuration
MODE = "ingest_only"  # "ingest_only" | "full" | "train_only" | "smoke_test"

REPO_URL = "https://github.com/RandomPerson5571/ad_eeg.git"
REPO_BRANCH = "main"
PROJECT_DIR = "/content/ad_eeg"

# Google Drive folder that contains EEG_data/ (ingest modes)
# Example: /content/drive/MyDrive/EEG_Project/EEG_data
DRIVE_EEG_DATA_DIR = "/content/drive/MyDrive/EEG_Project/EEG_data"

# Stable folder for feature parquet + ingest log (resume across Colab sessions)
DRIVE_FEATURES_DIR = "/content/drive/MyDrive/EEG_Project/features"

# Timestamped run outputs (metrics, models) copied here after training
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/EEG_Project/colab_outputs"

# Zenodo record ID — optional fallback for train_only if Drive features missing
ZENODO_RECORD_ID = None  # e.g. 1234567

TRAIN_MODELS = "xgboost,mlp"  # comma-separated: xgboost,mlp

In [ ]:
# @title Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# @title Clone repo and install dependencies
import os
import subprocess
import sys


def run(cmd, cwd=None):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, cwd=cwd)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {cmd}")


if os.path.exists(PROJECT_DIR):
    run(f"git -C {PROJECT_DIR} pull")
else:
    run(f"git clone --branch {REPO_BRANCH} {REPO_URL} {PROJECT_DIR}")

run(f"{sys.executable} -m pip install -q -r requirements.txt", cwd=PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# @title Link raw EEG from Drive (ingest modes)
import os
from pathlib import Path

INGEST_MODES = ("ingest_only", "full", "smoke_test")
eeg_link = Path(PROJECT_DIR) / "EEG_data"

if MODE in INGEST_MODES:
    drive_eeg = Path(DRIVE_EEG_DATA_DIR)
    if not drive_eeg.exists():
        raise FileNotFoundError(
            f"EEG data not found at {drive_eeg}.\n"
            "Upload the Miltiadous dataset so Drive contains:\n"
            "  EEG_data/dataset2/participants.tsv\n"
            "  EEG_data/dataset2/sub-001/eeg/...\n"
            "  EEG_data/dataset3/..."
        )

    if eeg_link.is_symlink() or eeg_link.exists():
        if eeg_link.is_symlink():
            eeg_link.unlink()
        elif eeg_link.is_dir():
            pass  # already a real folder in Colab
        else:
            eeg_link.unlink()

    if not eeg_link.exists():
        os.symlink(drive_eeg, eeg_link)

    !python scripts/download_data.py
else:
    print(f"Skipping EEG link (MODE={MODE})")

In [ ]:
# @title Run pipeline (pipeline.py)
import json
import os
import subprocess
import sys
from pathlib import Path

project = Path(PROJECT_DIR)
os.chdir(project)
sys.path.insert(0, str(project))


def run_py(args):
    cmd = [sys.executable, "-u"] + args
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


STAGES_BY_MODE = {
    "ingest_only": "preprocess,features",
    "train_only": "train",
    "smoke_test": "preprocess,features,train",
    "full": "preprocess,features,train",
}

if MODE not in STAGES_BY_MODE:
    raise ValueError(f"Unknown MODE: {MODE}")

stages = STAGES_BY_MODE[MODE]
pipeline_args = [
    "pipeline.py",
    "--dataset", "all",
    "--experiment", "baseline",
    "--stages", stages,
    "--model", TRAIN_MODELS,
]
if MODE == "smoke_test":
    pipeline_args.extend(["--limit", "2"])

run_py(pipeline_args)

# Show benchmark / metrics if training ran
if "train" in stages:
  for p in sorted((project / "data" / "results").rglob("benchmark.csv")):
      print(f"\n=== {p.relative_to(project)} ===", flush=True)
      print(p.read_text(), flush=True)
  for p in sorted((project / "data" / "results").rglob("metrics.json")):
      print(f"\n=== {p.relative_to(project)} ===", flush=True)
      print(json.dumps(json.loads(p.read_text()), indent=2), flush=True)

In [ ]:
# @title Save training outputs to Google Drive
import shutil
import sys
from datetime import datetime
from pathlib import Path

project = Path(PROJECT_DIR)
sys.path.insert(0, str(project))
from config import DATASETS  # noqa: E402

feature_artifacts = [
    "results/ingest_log.json",
    "results/preprocessing_config.json",
    *(f"parquet_files/features_dataset{ds}.parquet" for ds in DATASETS),
]

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_root = Path(DRIVE_OUTPUT_DIR) / f"run_{stamp}_{MODE}"
out_root.mkdir(parents=True, exist_ok=True)

artifacts = [
    *feature_artifacts,
    *[f"results/metrics_dataset{ds}.json" for ds in DATASETS],
    *[f"results/metrics_xgboost_dataset{ds}.json" for ds in DATASETS],
    *[f"results/metrics_mlp_dataset{ds}.json" for ds in DATASETS],
    *[f"results/subject_splits_dataset{ds}.json" for ds in DATASETS],
    *[f"classifier_models/saved_models/xgboost_eeg_classifier_dataset{ds}.joblib" for ds in DATASETS],
    *[f"classifier_models/saved_models/eeg_mlp_classifier_dataset{ds}.joblib" for ds in DATASETS],
]

copied = []
for rel in artifacts:
    src = project / rel
    if not src.exists():
        continue
    dest = out_root / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dest)
    copied.append(rel)

print(f"Saved {len(copied)} file(s) to:\n  {out_root}")
for rel in copied:
    print(f"  - {rel}")

## Google Drive layout

Upload the downloaded Miltiadous dataset so your Drive looks like:

```
MyDrive/
  EEG_Project/
    EEG_data/
      dataset2/
        participants.tsv
        sub-001/eeg/sub-001_task-eyesclosed_eeg.set
        ...
      dataset3/
        participants.tsv
        sub-001/eeg/sub-001_task-photomark_eeg.set
        ...
    features/               # stable — parquet + ingest_log (auto-saved by ingest_only)
    colab_outputs/          # timestamped training runs
```

**Typical Colab workflow**
1. `ingest_only` — run repeatedly until all subjects complete (skips already-done subjects)
2. `train_only` — loads features from `features/`, trains models
3. Run the save cell to archive metrics and models to `colab_outputs/`